# 导数


## 1.标量对向量求导
y=x_1^2+x_2^2+x_3^2
x=[x_1,x_2,x_3]^T
y对x的导数 = [2x_1, 2x_2, 2x_3]^T

## 2.向量对向量求导
y=[y_1,y_2]^T
x=[x_1,x_2]^T
y对x的导数 = [偏 y_1/偏 x_1, 偏 y_1/偏 x_2; 偏 y_2/偏 x_1, 偏 y_2/偏 x_2]

## 3.标量对矩阵求导
y=x_1^2+x_2^2+x_3^2+x_4^2
x=[x_{ij}]^T
y对x的导数 = [2x_{ij}]^T


In [30]:
%matplotlib inline
import numpy as np
from matplotlib_inline import backend_inline
from d2l import torch as d2l


def f(x):
    return 3 * x ** 2 - 4 * x

In [31]:
def numerical_lim(f, x, h):
    return (f(x + h) - f(x)) / h

h = 0.1
for i in range(5):
    print(f'h={h:.5f}, numerical limit={numerical_lim(f, 1, h):.5f}')
    h *= 0.1

##f'(x)=6x-4,当 x=1 的时候，极限逼近于 2

h=0.10000, numerical limit=2.30000
h=0.01000, numerical limit=2.03000
h=0.00100, numerical limit=2.00300
h=0.00010, numerical limit=2.00030
h=0.00001, numerical limit=2.00003


requires_grad=True代表需要进行求导，x.grad  # 默认值是None

In [47]:
import torch

# 方式一：创建时直接指定
x = torch.arange(4.0, requires_grad=True)
# 方式二：先创建，后开启梯度追踪（即你代码中的写法）
# x = torch.arange(4.0)
# x.requires_grad_(True)

y = 2 * torch.dot(x, x) #2*sum(x^2)
z = 3 * torch.dot(x, x)

In [48]:
y.backward(retain_graph=True)     #计算 y 的所有梯度
print(x.grad)  # 显示关于 x 的梯度
z.backward()
print(x.grad)  # x.grad = x.grad + 新梯度

tensor([ 0.,  4.,  8., 12.])
tensor([ 0., 10., 20., 30.])


In [ ]:
# 在默认情况下，PyTorch会累积梯度，我们需要清除之前的值就使用以下函数
x.grad.zero_()

# 非标量的反向传播
直接进行backward()不知道是对哪一个未知数进行求导。

In [ ]:
y = x * x   #此时 y 是一个向量
# 下面等价于y.backward(torch.ones(len(x)))
y.sum().backward()

## 分离计算

有时，我们希望[**将某些计算移动到记录的计算图之外**]。
例如，假设`y`是作为`x`的函数计算的，而`z`则是作为`y`和`x`的函数计算的。
想象一下，我们想计算`z`关于`x`的梯度，但由于某种原因，希望将`y`视为一个常数，
并且只考虑到`x`在`y`被计算后发挥的作用。

这里可以分离`y`来返回一个新变量`u`，该变量与`y`具有相同的值，
但丢弃计算图中如何计算`y`的任何信息。
换句话说，梯度不会向后流经`u`到`x`。
因此，下面的反向传播函数计算`z=u*x`关于`x`的偏导数，同时将`u`作为常数处理，
而不是`z=x*x*x`关于`x`的偏导数。

In [49]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

tensor([True, True, True, True])

# 正向传播vs反向积累
- 正向传播：为每一个参数跑一遍全流程
- 反向积累：存下中间步骤梯度，用空间换时间。

因此深度学习输入参数巨大，正向传播会导致计算量爆炸。

# 自动微分是动态计算的，即使要通过Python的控制流，也能正确计算梯度。


In [52]:
#这里的函数只进行✖️系数操作，最后得到的肯定是 ka，k 为某个常数
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True)    #随机生成一个a
d = f(a)        #d=ka


In [53]:
d.backward()    #d对a的导数=k
a.grad == d / a    # 这里应该输出 k

tensor(True)